<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/16-gans-diffusion-flow-matching.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **GAN、扩散模型与 Flow Matching** {#gans-diffusion-flow-matching}

GAN、扩散模型与 flow matching 都会把简单随机源转化为结构化数据，但它们学习的对象不同。GAN 通过对抗式 critic 学习生成器；扩散模型学习逆转渐进破坏过程；flow matching 学习一个随时间变化的速度场，其 ODE 会把源分布输运到数据分布。

![GAN、扩散模型与 flow matching 使用不同训练信号和采样路径。](assets/dl16-family-map.svg){fig-align="center" width="78%" fig-alt="三个面板比较对抗式单次生成、迭代扩散去噪与 ODE flow matching。"}

*基于 [Generative Adversarial Nets](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)、[DDPM](https://arxiv.org/abs/2006.11239) 与 [Flow Matching](https://arxiv.org/abs/2210.02747) 绘制的原创综合图。*

可执行主线使用 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits 数据集](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，采用 CC BY 4.0 许可。全部模型共享固定的 70/15/15 划分。一个只在真实训练图像上训练的分类器提供统一的领域特征空间，用于测量条件保真度、置信度、覆盖度和最近训练样本距离。它只是探针，不是中立且普适的评估器。

<details>
<summary><strong>PyTorch：建立共享数据、划分与评估探针</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1616):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).flatten(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    indices, test_size=0.30, random_state=1616, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1616,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]
train_scaled = train_x * 2 - 1
val_scaled = val_x * 2 - 1
test_scaled = test_x * 2 - 1


class DigitProbe(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Linear(32, 10)

    def forward(self, images, return_features=False):
        features = self.features(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


seed_everything(1617)
probe = DigitProbe()
optimizer = torch.optim.AdamW(probe.parameters(), lr=3e-3, weight_decay=1e-4)
for _ in range(65):
    optimizer.zero_grad()
    loss = F.cross_entropy(probe(train_x), train_y)
    loss.backward()
    optimizer.step()
probe.eval()
with torch.no_grad():
    probe_accuracy = float((probe(test_x).argmax(1) == test_y).float().mean())


def audit_samples(images_01, intended_labels=None):
    images_01 = images_01.detach().clamp(0, 1)
    with torch.no_grad():
        logits, features = probe(images_01, return_features=True)
        probabilities = logits.softmax(1)
        nearest = torch.cdist(images_01, train_x).min(1).values
        real_features = probe(test_x, return_features=True)[1]
    rounded_unique = torch.unique(torch.round(images_01 * 8) / 8, dim=0).shape[0] / len(images_01)
    result = {
        "confidence": float(probabilities.max(1).values.mean()),
        "rounded unique ratio": rounded_unique,
        "nearest-train distance": float(nearest.mean()),
        "feature mean gap": float((features.mean(0) - real_features.mean(0)).norm()),
    }
    if intended_labels is not None:
        result["conditional accuracy"] = float((probabilities.argmax(1) == intended_labels).float().mean())
        result["covered predicted classes"] = int(probabilities.argmax(1).unique().numel())
    return result


assert all_images.shape == (1797, 64)
assert len(set(train_idx) & set(test_idx)) == 0
assert probe_accuracy > 0.90
print({"split": (len(train_x), len(val_x), len(test_x)), "probe accuracy": round(probe_accuracy, 3),
       "pixel range": (float(train_scaled.min()), float(train_scaled.max()))})
```

</details>

图像只有 8×8，因此生成样本用于展示目标和失败模式，而不是展示照片级合成。所有质量结论只适用于当前数据集、架构、随机种子与短程 CPU 训练预算。


### **对抗式生成** {#adversarial-generation}

GAN 包含生成器 $G_{\theta}(z,c)$ 与判别器 $D_{\phi}(x,c)$。生成器把噪声 $z\sim p(z)$ 以及可选条件 $c$ 映射为样本；判别器学习区分真实配对与生成配对的证据。两者都不提供显式逐点似然，critic 会把分布差异转化为生成器梯度。

![生成器与判别器在耦合博弈中接收相反的损失。](assets/dl16-gan-game.svg){fig-align="center" width="76%" fig-alt="噪声和条件进入生成器，真实和生成观测进入判别器，两种相反损失分别更新两个网络。"}

*依据 [Goodfellow 等人](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html) 绘制的原创机制图。*

训练过程不是对单个静态损失做普通最小化。$G$ 改变时，判别器任务会改变；$D$ 改变时，生成器梯度场也会改变。因此，即使书面目标不变，更新比例、优化器、归一化、容量与数据增强也会改变博弈。

<details>
<summary><strong>PyTorch：定义并训练一个条件对抗生成器</strong></summary>

```python
class ConditionalGenerator(nn.Module):
    def __init__(self, noise_dim=20, embedding_dim=10):
        super().__init__()
        self.noise_dim = noise_dim
        self.label_embedding = nn.Embedding(10, embedding_dim)
        self.network = nn.Sequential(
            nn.Linear(noise_dim + embedding_dim, 96), nn.LeakyReLU(0.2),
            nn.Linear(96, 128), nn.LeakyReLU(0.2), nn.Linear(128, 64), nn.Tanh(),
        )

    def forward(self, noise, labels):
        return self.network(torch.cat([noise, self.label_embedding(labels)], dim=1))


class ConditionalDiscriminator(nn.Module):
    def __init__(self, embedding_dim=10):
        super().__init__()
        self.label_embedding = nn.Embedding(10, embedding_dim)
        self.network = nn.Sequential(
            nn.Linear(64 + embedding_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 64), nn.LeakyReLU(0.2), nn.Linear(64, 1),
        )

    def forward(self, images, labels):
        return self.network(torch.cat([images, self.label_embedding(labels)], dim=1)).squeeze(1)


seed_everything(1620)
gan_generator = ConditionalGenerator()
gan_discriminator = ConditionalDiscriminator()
g_optimizer = torch.optim.Adam(gan_generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(gan_discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
generator = torch.Generator().manual_seed(1620)
gan_trace = []
for step in range(850):
    batch_indices = torch.randint(len(train_scaled), (128,), generator=generator)
    real, labels = train_scaled[batch_indices], train_y[batch_indices]
    noise = torch.randn((128, gan_generator.noise_dim), generator=generator)
    fake = gan_generator(noise, labels)

    d_optimizer.zero_grad()
    real_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(real, labels), torch.full((128,), 0.9)
    )
    fake_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(fake.detach(), labels), torch.zeros(128)
    )
    discriminator_loss = real_loss + fake_loss
    discriminator_loss.backward()
    d_optimizer.step()

    g_optimizer.zero_grad()
    generator_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(fake, labels), torch.ones(128)
    )
    generator_loss.backward()
    g_optimizer.step()
    if step % 100 == 0:
        gan_trace.append((step, float(discriminator_loss.detach()), float(generator_loss.detach())))

evaluation_labels = torch.arange(10).repeat_interleave(20)
with torch.no_grad():
    gan_noise = torch.randn((len(evaluation_labels), gan_generator.noise_dim),
                            generator=torch.Generator().manual_seed(1621))
    gan_samples_scaled = gan_generator(gan_noise, evaluation_labels)
gan_samples = (gan_samples_scaled + 1) / 2
assert gan_samples.shape == (200, 64)
print({"final discriminator loss": round(float(discriminator_loss.detach()), 3),
       "final generator loss": round(float(generator_loss.detach()), 3),
       "recorded checkpoints": len(gan_trace)})
```

</details>

生成张量范围与有限损失只是实现检查。判别器损失接近 $\log 4$ 可能意味着平衡、欠拟合或恰好混淆，并不是质量分数；还必须检查样本和覆盖度。


### **GAN Minimax 目标** {#gan-minimax-objective}

原始博弈为

$$
\min_G\max_D\;V(D,G)=
\mathbb{E}_{x\sim p_{\text{data}}}\log D(x)
+\mathbb{E}_{z\sim p(z)}\log(1-D(G(z))).
$$

固定 $G$ 时，最优判别器为

$$
D^{*}(x)=\frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)}.
$$

代回后会得到相差常数的 Jensen-Shannon divergence 目标。该理论结果假设判别器达到最优且函数族无限制；有限神经网络的交替更新并不满足这些条件。

训练早期，较强判别器使 $D(G(z))\approx0$。Minimax 生成器损失 $\log(1-D(G(z)))$ 此时会饱和。Non-saturating 替代目标 $-\log D(G(z))$ 具有相同理想固定点，却提供更强梯度。

![Saturating 与 non-saturating 生成器损失在训练早期具有不同梯度。](assets/dl16-minimax-gradients.svg){fig-align="center" width="76%" fig-alt="两条损失曲线比较 saturating 目标的梯度消失与 non-saturating 目标的更强梯度。"}

*依据 [GAN 原始论文](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html) 推导的原创梯度对比图。*

<details>
<summary><strong>PyTorch：在已训练判别器上比较生成器梯度大小</strong></summary>

```python
gan_discriminator.eval()
demo_noise = torch.randn((128, gan_generator.noise_dim), generator=torch.Generator().manual_seed(1630))
demo_labels = train_y[:128]
demo_fake = gan_generator(demo_noise, demo_labels)
demo_logits = gan_discriminator(demo_fake, demo_labels)

saturating_loss = -F.binary_cross_entropy_with_logits(demo_logits, torch.zeros_like(demo_logits))
non_saturating_loss = F.binary_cross_entropy_with_logits(demo_logits, torch.ones_like(demo_logits))
saturating_gradient = torch.autograd.grad(saturating_loss, demo_fake, retain_graph=True)[0].norm(dim=1).mean()
non_saturating_gradient = torch.autograd.grad(non_saturating_loss, demo_fake)[0].norm(dim=1).mean()

assert saturating_gradient > 0 and non_saturating_gradient > 0
print({"mean D(fake)": round(float(demo_logits.sigmoid().mean().detach()), 3),
       "saturating input-gradient norm": round(float(saturating_gradient.detach()), 5),
       "non-saturating input-gradient norm": round(float(non_saturating_gradient.detach()), 5)})
```

</details>

梯度大小取决于当前 logits 和参数化。Non-saturating loss 改善了信号，但不会独自解决循环、mode collapse、过拟合或较差条件控制。


### **Conditional GAN 与架构改进** {#conditional-gans-architectural-improvements}

Conditional GAN 把 $c$ 提供给两个参与者：

$$
G(z,c)\rightarrow x,\qquad D(x,c)\rightarrow \mathbb{R}.
$$

判别器既要拒绝不真实图像，也要拒绝图像与条件不匹配的配对。[Conditional GAN](https://arxiv.org/abs/1411.1784) 最初展示了类别控制的数字生成。现代实现还可能采用 projection discriminator、conditional normalization、attention、残差块、spectral normalization，以及匹配良好的卷积上采样与下采样。

![生成器和判别器都接收条件，标签干预用于检查条件是否被使用。](assets/dl16-conditional-gan.svg){fig-align="center" width="76%" fig-alt="噪声与标签进入生成器，生成图像与标签进入判别器，审计仅改变标签。"}

*依据 [Mirza 与 Osindero](https://arxiv.org/abs/1411.1784) 绘制的原创条件契约图。*

<details>
<summary><strong>PyTorch：固定生成噪声并干预标签</strong></summary>

```python
gan_generator.eval()
fixed_noise = torch.randn((10, gan_generator.noise_dim), generator=torch.Generator().manual_seed(1640))
label_order_a = torch.arange(10)
label_order_b = torch.roll(label_order_a, shifts=1)
with torch.no_grad():
    samples_a = (gan_generator(fixed_noise, label_order_a) + 1) / 2
    samples_b = (gan_generator(fixed_noise, label_order_b) + 1) / 2
    intervention_distance = (samples_a - samples_b).pow(2).mean(dim=1).sqrt()
    predicted_a = probe(samples_a).argmax(1)
    predicted_b = probe(samples_b).argmax(1)

assert float(intervention_distance.mean()) > 0
print({"mean image change after label intervention": round(float(intervention_distance.mean()), 3),
       "predictions changed": int((predicted_a != predicted_b).sum()),
       "out of": len(predicted_a)})
```

</details>

输出随 $c$ 变化只能证明敏感性，不能证明控制正确。还必须测量条件准确率、条件内部多样性和 condition leakage。在 text-to-image 系统中，这会扩展为 prompt adherence、组合性和非预期相关性。


### **Wasserstein GAN** {#wasserstein-gans}

当数据分布与生成器分布位于较薄且不相交的流形上时，原始 GAN 使用的 divergence 可能给出较差梯度。WGAN 用实值 1-Lipschitz critic 替代概率判别器，并优化 Kantorovich-Rubinstein 对偶：

$$
W_1(p_r,p_g)=\sup_{\lVert f\rVert_L\le1}
\mathbb{E}_{p_r}[f(x)]-\mathbb{E}_{p_g}[f(x)].
$$

[WGAN](https://arxiv.org/abs/1701.07875) 最初使用权重裁剪执行约束。[WGAN-GP](https://papers.nips.cc/paper_files/paper/2017/hash/892c3b1c6dccd52936e27cbd0ff683d6-Abstract.html) 在真实样本与生成样本之间的插值点上惩罚 critic 梯度范数：

$$
\lambda\,\mathbb{E}_{\hat{x}}(\lVert\nabla_{\hat{x}}D(\hat{x})\rVert_2-1)^2.
$$

![1-Lipschitz critic 提供类似距离的信号，梯度惩罚施加于插值点。](assets/dl16-wgan.svg){fig-align="center" width="76%" fig-alt="真实与生成分布通过 1-Lipschitz critic 相连，critic 的输入梯度范数受到惩罚。"}

*依据 [WGAN](https://arxiv.org/abs/1701.07875) 与 [WGAN-GP](https://papers.nips.cc/paper_files/paper/2017/hash/892c3b1c6dccd52936e27cbd0ff683d6-Abstract.html) 绘制的原创结构图。*

<details>
<summary><strong>PyTorch：在相同划分上训练条件 WGAN-GP</strong></summary>

```python
class ConditionalCritic(ConditionalDiscriminator):
    pass


def gradient_penalty(critic, real, fake, labels, generator):
    interpolation = torch.rand((len(real), 1), generator=generator)
    mixed = (interpolation * real + (1 - interpolation) * fake).requires_grad_(True)
    score = critic(mixed, labels)
    gradient = torch.autograd.grad(score.sum(), mixed, create_graph=True)[0]
    return (gradient.norm(2, dim=1) - 1).pow(2).mean()


seed_everything(1650)
wgan_generator = ConditionalGenerator()
wgan_critic = ConditionalCritic()
wg_optimizer = torch.optim.Adam(wgan_generator.parameters(), lr=1e-4, betas=(0.0, 0.9))
wc_optimizer = torch.optim.Adam(wgan_critic.parameters(), lr=1e-4, betas=(0.0, 0.9))
generator = torch.Generator().manual_seed(1650)
for generator_step in range(360):
    for _ in range(3):
        batch_indices = torch.randint(len(train_scaled), (128,), generator=generator)
        real, labels = train_scaled[batch_indices], train_y[batch_indices]
        noise = torch.randn((128, wgan_generator.noise_dim), generator=generator)
        fake = wgan_generator(noise, labels).detach()
        wc_optimizer.zero_grad()
        penalty = gradient_penalty(wgan_critic, real, fake, labels, generator)
        critic_loss = wgan_critic(fake, labels).mean() - wgan_critic(real, labels).mean() + 10 * penalty
        critic_loss.backward()
        wc_optimizer.step()
    labels = train_y[torch.randint(len(train_y), (128,), generator=generator)]
    noise = torch.randn((128, wgan_generator.noise_dim), generator=generator)
    wg_optimizer.zero_grad()
    wgan_loss = -wgan_critic(wgan_generator(noise, labels), labels).mean()
    wgan_loss.backward()
    wg_optimizer.step()

with torch.no_grad():
    wgan_noise = torch.randn((len(evaluation_labels), wgan_generator.noise_dim),
                             generator=torch.Generator().manual_seed(1651))
    wgan_samples = (wgan_generator(wgan_noise, evaluation_labels) + 1) / 2

assert wgan_samples.shape == gan_samples.shape
print({"critic loss": round(float(critic_loss.detach()), 3),
       "gradient penalty": round(float(penalty.detach()), 3),
       "generator loss": round(float(wgan_loss.detach()), 3)})
```

</details>

训练后的 critic 目标并不自动成为准确的 $W_1$ 数值估计：有限容量、不完全优化和近似 Lipschitz 约束都会产生影响。它的实际价值在于生成器梯度与训练诊断，而不是保证成立的输运证书。


### **Mode Collapse 与训练不稳定** {#mode-collapse-training-instability}

Mode collapse 会把许多 $z$ 映射到少量有效输出。局部样本可能具有较高保真度，但生成器会遗漏其他数据模式。振荡源于两个参与者不断追逐变化的对手；判别器过拟合可能暴露无用方向；梯度消失或爆炸则可能停止学习。

![Mode collapse 可以保留局部真实性，却失去分布覆盖。](assets/dl16-mode-collapse.svg){fig-align="center" width="76%" fig-alt="多个数据模式与只覆盖一个模式的生成器进行比较，诊断分别衡量保真度、覆盖度与记忆。"}

*原创失败模式图。*

常见缓解方法包括平衡容量和更新比例、non-saturating 或 Wasserstein 目标、梯度惩罚、spectral normalization、minibatch feature、数据增强与多个生成器。但任何方法都不能取代跨条件和随机种子的样本检查。

<details>
<summary><strong>PyTorch：审计 GAN 保真度、覆盖度、多样性与复制</strong></summary>

```python
gan_audit = audit_samples(gan_samples, evaluation_labels)
wgan_audit = audit_samples(wgan_samples, evaluation_labels)
for name, row in {"non-saturating GAN": gan_audit, "WGAN-GP": wgan_audit}.items():
    print({name: {key: round(value, 3) if isinstance(value, float) else value
                  for key, value in row.items()}})

assert 0 <= gan_audit["conditional accuracy"] <= 1
assert 1 <= wgan_audit["covered predicted classes"] <= 10
```

</details>

探针可能对生成伪影表现得很自信却是错误的。量化后的唯一率可能漏掉语义坍塌，最近像素距离也可能漏掉轻微变换后的记忆。可靠研究还需要重复随机种子、类别条件 precision/recall、特征空间覆盖、训练数据提取测试和人工审查。


### **扩散前向过程** {#diffusion-forward-process}

DDPM 定义固定 Markov 破坏过程：

$$
q(x_t\mid x_{t-1})=\mathcal{N}(\sqrt{1-\beta_t}\,x_{t-1},\beta_t I).
$$

令 $\alpha_t=1-\beta_t$、$\bar{\alpha}_t=\prod_{s=1}^{t}\alpha_s$，任意 timestep 都可以直接采样：

$$
x_t=\sqrt{\bar{\alpha}_t}x_0+\sqrt{1-\bar{\alpha}_t}\epsilon,
\qquad \epsilon\sim\mathcal{N}(0,I).
$$

![前向扩散逐渐降低信噪比，并允许直接采样任意 timestep。](assets/dl16-forward-diffusion.svg){fig-align="center" width="78%" fig-alt="干净样本变得越来越嘈杂，链旁展示闭式采样公式。"}

*依据 [DDPM](https://arxiv.org/abs/2006.11239) 绘制的原创流程图。*

噪声 schedule 控制信噪比（SNR）。终点噪声不足会留下训练与采样起点不匹配，早期加噪过强又会破坏有效学习信号。现代 schedule 经常通过 log SNR 而不是原始 $\beta_t$ 描述。

<details>
<summary><strong>PyTorch：构造并验证闭式前向过程</strong></summary>

```python
diffusion_steps = 40
betas = torch.linspace(1e-4, 0.14, diffusion_steps)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def extract(values, timesteps, target):
    return values[timesteps].view(-1, *([1] * (target.ndim - 1)))


def q_sample(clean, timesteps, noise):
    signal = extract(alpha_bars.sqrt(), timesteps, clean)
    noise_scale = extract((1 - alpha_bars).sqrt(), timesteps, clean)
    return signal * clean + noise_scale * noise


generator = torch.Generator().manual_seed(1660)
forward_clean = test_scaled[:180]
forward_noise = torch.randn(forward_clean.shape, generator=generator)
forward_statistics = {}
for timestep in (0, 9, 19, 39):
    t = torch.full((len(forward_clean),), timestep, dtype=torch.long)
    noisy = q_sample(forward_clean, t, forward_noise)
    forward_statistics[timestep] = {
        "alpha_bar": float(alpha_bars[timestep]),
        "correlation": float(torch.corrcoef(torch.stack([forward_clean.flatten(), noisy.flatten()]))[0, 1]),
        "std": float(noisy.std()),
    }
print({t: {k: round(v, 3) for k, v in row.items()} for t, row in forward_statistics.items()})
assert forward_statistics[0]["correlation"] > forward_statistics[39]["correlation"]
```

</details>

标准 DDPM 的前向过程不需要学习。其可处理的后验结构会在随机选择的噪声级别上产生监督式去噪目标。


### **学习反向去噪过程** {#learning-reverse-denoising-process}

精确反向条件 $q(x_{t-1}\mid x_t)$ 依赖未知数据分布。神经模型近似其均值，或预测等价的 noise/score 目标；所选方差 schedule 控制随机性。时间嵌入告诉网络当前处理的 SNR 区域，条件嵌入则提供类别、文本或其他控制信息。

为了后续使用 classifier-free guidance，同一个网络同时接受有条件和无条件训练。一部分标签会被替换为已学习的 null condition。这是有意的 condition dropout，而不是缺失数据泄漏。

<details>
<summary><strong>PyTorch：训练带 condition dropout 的条件多噪声去噪器</strong></summary>

```python
def time_embedding(timesteps, dimension=24):
    half = dimension // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half) / max(half - 1, 1))
    angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(0)
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class DiffusionDenoiser(nn.Module):
    def __init__(self, data_dim=64, hidden=128, label_dim=20):
        super().__init__()
        self.data_dim = data_dim
        self.null_label = 10
        self.label_embedding = nn.Embedding(11, label_dim)
        self.network = nn.Sequential(
            nn.Linear(data_dim + 24 + label_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, data_dim),
        )

    def forward(self, noisy, timesteps, labels):
        context = torch.cat([noisy, time_embedding(timesteps), self.label_embedding(labels)], dim=1)
        return self.network(context)


seed_everything(1670)
diffusion_model = DiffusionDenoiser()
optimizer = torch.optim.AdamW(diffusion_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1670)
diffusion_loss_trace = []
for step in range(1500):
    batch_indices = torch.randint(len(train_scaled), (192,), generator=generator)
    clean, labels = train_scaled[batch_indices], train_y[batch_indices].clone()
    timesteps = torch.randint(diffusion_steps, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    noisy = q_sample(clean, timesteps, noise)
    drop_condition = torch.rand(len(clean), generator=generator) < 0.15
    labels[drop_condition] = diffusion_model.null_label
    optimizer.zero_grad()
    predicted_noise = diffusion_model(noisy, timesteps, labels)
    diffusion_loss = F.mse_loss(predicted_noise, noise)
    diffusion_loss.backward()
    torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
    optimizer.step()
    if step % 150 == 0:
        diffusion_loss_trace.append(float(diffusion_loss.detach()))

diffusion_model.eval()
with torch.no_grad():
    val_t = torch.randint(diffusion_steps, (len(val_scaled),), generator=torch.Generator().manual_seed(1671))
    val_noise = torch.randn(val_scaled.shape, generator=torch.Generator().manual_seed(1672))
    val_noisy = q_sample(val_scaled, val_t, val_noise)
    val_noise_mse = float(F.mse_loss(diffusion_model(val_noisy, val_t, val_y), val_noise))

assert val_noise_mse < 1.0
print({"validation noise MSE": round(val_noise_mse, 3),
       "first recorded loss": round(diffusion_loss_trace[0], 3),
       "last recorded loss": round(diffusion_loss_trace[-1], 3)})
```

</details>

均匀 timestep sampling 对所有索引赋予相同权重，却不会对所有 SNR 区域赋予相同权重。Loss weighting、架构、self-conditioning、学习方差与数据参数化都会改变模型容量被分配的位置。


### **DDPM 目标与参数化** {#ddpm-objectives-parameterizations}

简化 DDPM 目标预测采样噪声：

$$
\mathcal{L}_{\epsilon}=\mathbb{E}_{x_0,t,\epsilon}
\lVert\epsilon-\epsilon_{\theta}(x_t,t,c)\rVert_2^2.
$$

等价目标包括干净数据 $x_0$ 与 velocity：

$$
v_t=\sqrt{\bar{\alpha}_t}\epsilon-
\sqrt{1-\bar{\alpha}_t}x_0.
$$

![当 schedule 已知时，noise、clean-data 与 velocity prediction 可通过代数互相转换。](assets/dl16-reverse-diffusion.svg){fig-align="center" width="78%" fig-alt="去噪器接收 noisy data、timestep 和 condition，再预测 epsilon、x0 或 velocity。"}

*依据 [DDPM](https://arxiv.org/abs/2006.11239) 绘制的原创参数化图。*

虽然参数化可以代数转换，但它们在不同 SNR 下的优化尺度不同。裁剪 $x_0$ 估计、按 SNR 加权和预测方差都会改变实际模型。“相同目标”应表示权重与 schedule 都相同，而不只是目标可以转换。

<details>
<summary><strong>PyTorch：验证任意 timestep 上 epsilon、x0 与 v 的转换</strong></summary>

```python
generator = torch.Generator().manual_seed(1680)
parameter_clean = test_scaled[:96]
parameter_t = torch.randint(diffusion_steps, (len(parameter_clean),), generator=generator)
parameter_noise = torch.randn(parameter_clean.shape, generator=generator)
parameter_noisy = q_sample(parameter_clean, parameter_t, parameter_noise)
a = extract(alpha_bars.sqrt(), parameter_t, parameter_clean)
b = extract((1 - alpha_bars).sqrt(), parameter_t, parameter_clean)
velocity = a * parameter_noise - b * parameter_clean
clean_from_velocity = a * parameter_noisy - b * velocity
noise_from_velocity = b * parameter_noisy + a * velocity
clean_from_noise = (parameter_noisy - b * parameter_noise) / a

assert torch.allclose(clean_from_velocity, parameter_clean, atol=2e-5)
assert torch.allclose(noise_from_velocity, parameter_noise, atol=2e-5)
assert torch.allclose(clean_from_noise, parameter_clean, atol=2e-5)
print({"max x0 reconstruction error": float((clean_from_velocity - parameter_clean).abs().max()),
       "max epsilon reconstruction error": float((noise_from_velocity - parameter_noise).abs().max())})
```

</details>

转换 checkpoint 或 scheduler 时，这些恒等式十分重要。错误的 prediction type 可能让早期去噪看似合理，却在后期严重发散。


### **Score-Based Model、SDE 与 Probability-Flow ODE** {#score-sde-probability-flow-ode}

对于 variance-preserving diffusion，epsilon predictor 给出 score 估计：

$$
s_{\theta}(x_t,t)\approx\nabla_{x_t}\log p_t(x_t)
=-\frac{\epsilon_{\theta}(x_t,t)}{\sqrt{1-\bar{\alpha}_t}}.
$$

连续时间中，前向 SDE $dx=f(x,t)dt+g(t)dW_t$ 的反向 SDE 为

$$
dx=[f(x,t)-g(t)^2\nabla_x\log p_t(x)]dt+g(t)d\bar{W}_t,
$$

并从噪声向数据积分。Probability-flow ODE 移除 Brownian 项，并使用一半 score 修正。Score 精确时，两者具有相同边缘分布，但单条路径和数值误差不同。

![反向 SDE 是随机过程，probability-flow ODE 对固定初始噪声则是确定过程。](assets/dl16-sde-ode.svg){fig-align="center" width="78%" fig-alt="随机反向 SDE 轨迹与确定 probability-flow ODE 轨迹在精确 score 下共享时间边缘分布。"}

*依据 [Score-Based Generative Modeling through SDEs](https://openreview.net/pdf?id=PxTIG12RRHS) 绘制的原创对比图。*

<details>
<summary><strong>PyTorch：把 noise prediction 转为 score，并区分漂移与随机项</strong></summary>

```python
diffusion_model.eval()
generator = torch.Generator().manual_seed(1690)
sde_clean = test_scaled[:128]
sde_t = torch.full((len(sde_clean),), 24, dtype=torch.long)
sde_noise = torch.randn(sde_clean.shape, generator=generator)
sde_noisy = q_sample(sde_clean, sde_t, sde_noise)
with torch.no_grad():
    predicted_epsilon = diffusion_model(sde_noisy, sde_t, test_y[:128])
noise_scale = extract((1 - alpha_bars).sqrt(), sde_t, sde_noisy)
estimated_score = -predicted_epsilon / noise_scale

beta = betas[24]
forward_drift = -0.5 * beta * sde_noisy
reverse_sde_drift = forward_drift - beta * estimated_score
probability_flow_drift = forward_drift - 0.5 * beta * estimated_score
stochastic_increment = math.sqrt(float(beta)) * torch.randn(
    sde_noisy.shape, generator=torch.Generator().manual_seed(1691)
)

assert torch.allclose(reverse_sde_drift - forward_drift,
                      2 * (probability_flow_drift - forward_drift), atol=1e-6)
print({"mean score norm": round(float(estimated_score.norm(dim=1).mean()), 3),
       "reverse drift norm": round(float(reverse_sde_drift.norm(dim=1).mean()), 3),
       "stochastic increment std": round(float(stochastic_increment.std()), 3)})
```

</details>

代码只检查局部代数，不是连续时间求解器。SDE/ODE 似然与采样需要一致的连续 schedule、数值积分器和误差控制。若不匹配约定就直接复用离散 DDPM 网络，相关解释可能失效。


### **Classifier Guidance 与 Classifier-Free Guidance** {#classifier-classifier-free-guidance}

Classifier guidance 把噪声感知分类器提供的 $\nabla_{x_t}\log p(c\mid x_t)$ 加入无条件 score。它需要一个跨噪声级别训练的独立分类器，也可能利用分类器错误。

Classifier-free guidance（CFG）让一个去噪器同时在有条件和无条件模式下训练，再组合预测：

$$
\epsilon_{\mathrm{cfg}}=epsilon_{u}
+w(\epsilon_{c}-\epsilon_{u}).
$$

$w=0$ 表示无条件，$w=1$ 是普通条件预测，$w>1$ 则朝条件方向外推。更强 guidance 通常改善条件遵循与表面保真度，却会降低多样性或导致过饱和。

![Classifier-free guidance 使用可调外推尺度组合无条件与条件预测。](assets/dl16-guidance.svg){fig-align="center" width="78%" fig-alt="无条件与条件 epsilon prediction 组合成由尺度 w 控制的 guided prediction。"}

*依据 [Ho 与 Salimans](https://openreview.net/pdf/ea628d03c92a49b54bc2d757d209e024e7885980.pdf) 绘制的原创 CFG 图。*

<details>
<summary><strong>PyTorch：实现带 CFG 的确定性 DDIM 采样</strong></summary>

```python
@torch.no_grad()
def guided_epsilon(model, noisy, timestep, labels, guidance_scale):
    t = torch.full((len(noisy),), timestep, dtype=torch.long)
    conditional = model(noisy, t, labels)
    null_labels = torch.full_like(labels, model.null_label)
    unconditional = model(noisy, t, null_labels)
    return unconditional + guidance_scale * (conditional - unconditional)


@torch.no_grad()
def ddim_sample(model, labels, step_count=20, guidance_scale=1.0, seed=1700, data_dim=64):
    generator = torch.Generator().manual_seed(seed)
    sample = torch.randn((len(labels), data_dim), generator=generator)
    schedule = torch.linspace(diffusion_steps - 1, 0, step_count).round().long().unique_consecutive()
    for index, timestep_tensor in enumerate(schedule):
        timestep = int(timestep_tensor)
        epsilon = guided_epsilon(model, sample, timestep, labels, guidance_scale)
        alpha_bar = alpha_bars[timestep]
        predicted_clean = ((sample - torch.sqrt(1 - alpha_bar) * epsilon) /
                           torch.sqrt(alpha_bar)).clamp(-1, 1)
        next_timestep = int(schedule[index + 1]) if index + 1 < len(schedule) else -1
        next_alpha_bar = alpha_bars[next_timestep] if next_timestep >= 0 else torch.tensor(1.0)
        sample = torch.sqrt(next_alpha_bar) * predicted_clean + torch.sqrt(1 - next_alpha_bar) * epsilon
    return sample


guidance_samples = {}
for offset, scale in enumerate((0.0, 1.0, 3.0)):
    generated = ddim_sample(diffusion_model, evaluation_labels, step_count=20,
                            guidance_scale=scale, seed=1700)
    guidance_samples[scale] = (generated + 1) / 2
    row = audit_samples(guidance_samples[scale], evaluation_labels)
    print({"guidance": scale, **{key: round(value, 3) if isinstance(value, float) else value
                                 for key, value in row.items()}})

assert all(samples.shape == (200, 64) for samples in guidance_samples.values())
```

</details>

CFG 是部署 sampler 的组成部分，因此必须对尺度与 schedule 做版本管理。最佳数值取决于条件类型、模型、negative prompt 或 null embedding，以及评估目标。


### **Latent Diffusion** {#latent-diffusion}

Pixel-space diffusion 会在完整空间分辨率上反复执行大型网络。Latent diffusion 先学习编码器 $E$ 与解码器 $D$，再建模 $z=E(x)$：

$$
x\xrightarrow{E}z,\qquad z_T\rightarrow\cdots\rightarrow z_0,
\qquad \hat{x}=D(z_0).
$$

![Latent diffusion 把重复去噪移动到编码器与解码器之间的压缩表示。](assets/dl16-latent-diffusion.svg){fig-align="center" width="78%" fig-alt="图像先编码，在较小潜空间迭代去噪，再解码回像素。"}

*依据 [Latent Diffusion Models](https://arxiv.org/abs/2112.10752) 绘制的原创架构图。*

压缩会减少计算并突出感知结构，但生成器无法恢复自编码器丢弃的信息。重构质量、latent scaling、解码器伪影与 diffusion 质量是彼此独立的失败来源。

<details>
<summary><strong>PyTorch：训练紧凑自编码器及其潜空间扩散模型</strong></summary>

```python
class LatentAutoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.SiLU(), nn.Linear(48, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.SiLU(), nn.Linear(48, 64), nn.Tanh())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


seed_everything(1710)
latent_autoencoder = LatentAutoencoder()
optimizer = torch.optim.AdamW(latent_autoencoder.parameters(), lr=2e-3, weight_decay=1e-5)
for _ in range(75):
    optimizer.zero_grad()
    reconstruction, _ = latent_autoencoder(train_scaled)
    reconstruction_loss = F.mse_loss(reconstruction, train_scaled)
    reconstruction_loss.backward()
    optimizer.step()
latent_autoencoder.eval()
with torch.no_grad():
    train_latent_raw = latent_autoencoder.encoder(train_scaled)
    test_reconstruction = latent_autoencoder(test_scaled)[0]
latent_mean = train_latent_raw.mean(0)
latent_std = train_latent_raw.std(0).clamp_min(1e-5)
train_latent = (train_latent_raw - latent_mean) / latent_std

latent_diffusion_model = DiffusionDenoiser(data_dim=16, hidden=96)
optimizer = torch.optim.AdamW(latent_diffusion_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1710)
for _ in range(900):
    batch_indices = torch.randint(len(train_latent), (192,), generator=generator)
    clean, labels = train_latent[batch_indices], train_y[batch_indices].clone()
    timesteps = torch.randint(diffusion_steps, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    noisy = q_sample(clean, timesteps, noise)
    labels[torch.rand(len(labels), generator=generator) < 0.15] = latent_diffusion_model.null_label
    optimizer.zero_grad()
    loss = F.mse_loss(latent_diffusion_model(noisy, timesteps, labels), noise)
    loss.backward()
    optimizer.step()

latent_generated = ddim_sample(latent_diffusion_model, evaluation_labels, step_count=20,
                               guidance_scale=1.5, seed=1711, data_dim=16)
with torch.no_grad():
    latent_unscaled = latent_generated * latent_std + latent_mean
    latent_samples = (latent_autoencoder.decoder(latent_unscaled) + 1) / 2
latent_audit = audit_samples(latent_samples, evaluation_labels)
print({"autoencoder test MSE": round(float(F.mse_loss(test_reconstruction, test_scaled)), 4),
       "pixel dimension": 64, "latent dimension": 16,
       **{key: round(value, 3) if isinstance(value, float) else value for key, value in latent_audit.items()}})
assert latent_samples.shape == (200, 64)
```

</details>

本章的紧凑自编码器只用于教学。生产级 latent diffusion 通常使用感知与对抗重构目标、空间 latent tensor 和经过校准的缩放。计算节省应包含编码器/解码器成本与内存，而不只比较潜变量维度。


### **扩散采样与加速** {#diffusion-sampling-acceleration}

祖先 DDPM 采样在每个反向步骤加入后验噪声。DDIM 构造与训练目标相同的非 Markov 过程，并允许确定性轨迹。对 timestep 做子采样会减少神经函数评估次数（NFE），但粗糙积分会累积模型误差与离散误差。

加速方法包括高阶 ODE/SDE solver、timestep 优化、progressive distillation、consistency model、latent-space generation 与缓存。实际 wall-clock speed 还取决于网络架构、batch size、内存移动与硬件；NFE 不是完整延迟指标。

<details>
<summary><strong>PyTorch：在同一模型下比较完整与缩减 DDIM schedule</strong></summary>

```python
accelerated_results = {}
for offset, steps in enumerate((40, 20, 8)):
    samples_scaled = ddim_sample(diffusion_model, evaluation_labels, step_count=steps,
                                 guidance_scale=1.5, seed=1720)
    samples = (samples_scaled + 1) / 2
    accelerated_results[steps] = audit_samples(samples, evaluation_labels)
    print({"requested NFE": steps,
           **{key: round(value, 3) if isinstance(value, float) else value
              for key, value in accelerated_results[steps].items()}})

assert set(accelerated_results) == {40, 20, 8}
```

</details>

相同初始噪声使其成为 timestep schedule 的配对比较。在短程训练模型中，较少步骤仍可能获得更高探针分数，因为重复且有偏的去噪更新会累积误差；这不能证明粗粒度采样普遍更好。严格 sampler benchmark 还需要重复随机种子、实测延迟、置信区间和任务特定质量指标。


### **Flow Matching 与 Rectified Flow** {#flow-matching-rectified-flow}

Continuous normalizing flow 定义

$$
\frac{dz_t}{dt}=v_{\theta}(z_t,t,c).
$$

Flow matching 在训练期间避免求解该 ODE。首先选择连接源 $x_0$ 与数据 $x_1$ 的条件概率路径。对于本章使用的直线插值，

$$
x_t=(1-t)x_0+tx_1,\qquad u_t=x_1-x_0,
$$

再训练 $v_{\theta}(x_t,t,c)$ 回归 $u_t$。生成时采样 $x_0$，并把学习到的场数值积分到 $t=1$。

![Flow matching 采样端点，在训练时无需模拟即可回归路径速度，再通过 ODE 积分生成。](assets/dl16-flow-matching.svg){fig-align="center" width="78%" fig-alt="噪声与数据端点定义线性路径和目标速度，生成阶段积分已学习 ODE。"}

*依据 [Flow Matching](https://arxiv.org/abs/2210.02747) 与 [Rectified Flow](https://openreview.net/pdf?id=gWxpdtQpiYV) 绘制的原创流程图。*

Rectified flow 强调直线路径，并可通过 reflow 进一步拉直学到的 coupling。“直线”描述的是某种 coupling 下的轨迹，并不保证有限模型可以一步生成高质量结果。

<details>
<summary><strong>PyTorch：训练条件 flow-matching 速度场并对其积分</strong></summary>

```python
class ConditionalVelocity(nn.Module):
    def __init__(self, data_dim=64, hidden=128):
        super().__init__()
        self.label_embedding = nn.Embedding(10, 20)
        self.network = nn.Sequential(
            nn.Linear(data_dim + 1 + 20, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, data_dim),
        )

    def forward(self, points, times, labels):
        return self.network(torch.cat([points, times, self.label_embedding(labels)], dim=1))


seed_everything(1730)
velocity_model = ConditionalVelocity()
optimizer = torch.optim.AdamW(velocity_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1730)
for _ in range(1200):
    batch_indices = torch.randint(len(train_scaled), (192,), generator=generator)
    data, labels = train_scaled[batch_indices], train_y[batch_indices]
    source = torch.randn(data.shape, generator=generator)
    times = torch.rand((len(data), 1), generator=generator)
    path_points = (1 - times) * source + times * data
    target_velocity = data - source
    optimizer.zero_grad()
    flow_loss = F.mse_loss(velocity_model(path_points, times, labels), target_velocity)
    flow_loss.backward()
    optimizer.step()


@torch.no_grad()
def flow_sample(model, labels, steps, seed):
    generator = torch.Generator().manual_seed(seed)
    points = torch.randn((len(labels), 64), generator=generator)
    dt = 1.0 / steps
    for step in range(steps):
        times = torch.full((len(labels), 1), (step + 0.5) * dt)
        points = points + dt * model(points, times, labels)
    return points


flow_results = {}
for offset, steps in enumerate((1, 5, 20)):
    flow_samples = (flow_sample(velocity_model, evaluation_labels, steps, 1731) + 1) / 2
    flow_results[steps] = audit_samples(flow_samples, evaluation_labels)
    print({"Euler steps": steps,
           **{key: round(value, 3) if isinstance(value, float) else value
              for key, value in flow_results[steps].items()}})

assert math.isfinite(float(flow_loss))
```

</details>

独立端点 coupling 会产生交叉条件路径，回归模型学习它们的条件平均速度。更好的 coupling、optimal-transport path、reflow 与高阶 solver 可以减少曲率或积分误差。训练 MSE 本身不衡量样本质量。


### **GAN、Diffusion 与 Flow Matching 对比** {#gans-diffusion-flow-matching-compared}

| 属性 | GAN | Diffusion / score model | Flow matching |
|---|---|---|---|
| 学习对象 | 通过 critic 学习 generator | denoiser、noise、score 或 velocity target | 随时间变化的 velocity field |
| 训练信号 | 对抗式分布比较 | 监督式破坏目标 | 监督式条件路径速度 |
| 似然 | 通常不可用 | 需要额外机制的下界或 ODE likelihood | 可通过 divergence integration 得到 CNF likelihood |
| 采样 | 通常一次 generator 前向传播 | 迭代反向 chain/SDE/ODE | 迭代 ODE 求解 |
| 条件控制 | 两个参与者都接收 $c$ | 条件 denoiser 与 guidance | 条件 velocity field 及其 guidance 变体 |
| 主要优势 | 采样快、输出锐利 | 回归稳定、覆盖较好、条件方式灵活 | simulation-free training 简洁、输运视角清晰 |
| 主要风险 | mode collapse 与博弈不稳定 | NFE 多、schedule/solver error | 路径/coupling 选择和 ODE 离散化 |

本章数值不是排行榜。每个模型使用了较小但不同的优化预算和架构。共享探针可以暴露失败维度，但它的置信度和特征会偏向从真实数字中学到的规律。公平比较需要匹配计算量、调优预算、样本数、重复随机种子，以及适合领域的评估器。

模型选择取决于系统需求。当单次采样延迟最重要且对抗训练可以管理时，GAN 仍然有吸引力；当覆盖度、条件控制、编辑与成熟工具比迭代成本更重要时，diffusion 更有优势；当直接速度回归和 ODE 输运能提供有用的质量-速度权衡时，flow matching 很合适。混合系统经常组合 latent autoencoder、对抗式重构、diffusion 或 flow prior，以及 distillation。


### **章节对比与总结** {#chapter-comparison-summary}

本章通过一个数据集和一套审计契约连接了三种现代生成范式。GAN 部分展示了双参与者优化、non-saturating gradient、条件干预、WGAN-GP 与模式覆盖诊断。Diffusion 部分推导直接加噪、训练一个跨 SNR 的条件去噪器、转换 epsilon/x0/v 参数化、连接离散扩散与 SDE/ODE 视角，并测试 CFG、潜空间压缩和缩减步数采样。Flow matching 则用选定路径上的速度回归替代反向噪声预测。

核心区别是所学习的场。GAN 从变化的 critic 接收梯度；diffusion model 跨噪声尺度学习与 score 等价的去噪场；flow matching 学习输运速度场。采样成本由此产生：一次 generator evaluation、多次反向去噪 evaluation，或者具有指定函数评估次数的 ODE solver。

实际审查应验证条件契约、数据缩放、schedule 或 path、prediction parameterization、sampler equation，以及模型与 solver 的兼容性。评估应分离保真度、覆盖度、条件遵循、新颖性、记忆与延迟，报告随机种子和计算量，并避免把分类器分数或漂亮样本网格当作分布质量证明。

第 17 章将从分布生成转向序列决策与 world model。本章的生成组件会成为 learned simulator、trajectory model、policy 或 imagination mechanism，但决策质量还取决于 reward、不确定性、探索与累积模型误差。
